# TP : Introduction à l'apprentissage automatique avec le risque de crédit (German Credit)

**Durée estimée :** 2 heures · **Niveau :** premier cycle avancé ou début de maîtrise, bases de Python et de statistique

## Objectifs

À la fin de ce TP, vous saurez :

1. identifier le **type d'apprentissage** d'un problème et justifier ce choix ;
2. charger un jeu de données public depuis une URL et l'explorer graphiquement ;
3. préparer les données (variables catégorielles, mise à l'échelle) sans fuite de données ;
4. entraîner et comparer plusieurs modèles de classification à partir d'une référence simple ;
5. évaluer un modèle avec une matrice de confusion, une courbe ROC et une courbe d'apprentissage ;
6. **traduire les erreurs en coûts financiers** et choisir un seuil de décision en conséquence ;
7. interpréter un modèle et discuter ses limites.

## 1. De quel type d'apprentissage s'agit-il ?

Le problème est le suivant : **à partir des caractéristiques d'un client et de son prêt (statut du compte, durée, montant, historique de crédit, âge, emploi, etc.), prédire s'il s'agit d'un bon ou d'un mauvais risque de crédit.**

Ce problème relève de l'**apprentissage supervisé**, plus précisément de la **classification binaire** :

- **Supervisé** : chaque dossier est associé à une étiquette connue (bon ou mauvais payeur, observé après coup). Le modèle apprend à partir de paires (entrée, sortie).
- **Classification binaire** : la cible n'a que deux valeurs. Si la cible était le montant du prêt, ce serait une **régression**.
- **À coûts asymétriques** : c'est ce qui distingue ce TP d'un problème de classification classique. Accorder un prêt à un mauvais payeur coûte beaucoup plus cher à la banque que refuser un bon client. **Toutes les erreurs ne se valent pas**, et le critère de choix du modèle doit en tenir compte.

Pour distinguer ce cas de l'apprentissage non supervisé, on se demande : **la bonne réponse est-elle connue pour chaque exemple ?** Ici, oui : l'issue du prêt est renseignée pour les 1 000 dossiers. Un algorithme de partitionnement ne recevrait pas cette information et chercherait seulement des groupes de clients ressemblants.

> **Remarque.** Une différence importante avec un TP de classification d'espèces : ici, l'exactitude (part de bonnes prédictions) est un critère **insuffisant**, voire trompeur. Nous le verrons dès la comparaison des modèles.

## 2. Le jeu de données

Le jeu de données **Statlog (German Credit Data)** (Hofmann, 1994) est hébergé par le dépôt UCI Machine Learning Repository (identifiant 144). Il décrit **1 000 demandes de crédit** d'une banque allemande, avec **20 variables** et une étiquette : 700 bons risques et **300 mauvais risques**.

| Variable | Type | Description |
|---|---|---|
| `class` (cible) | catégorielle | `good` (bon risque) ou `bad` (mauvais risque) |
| `checking_status` | catégorielle | Situation du compte courant |
| `duration` | numérique | Durée du crédit (mois) |
| `credit_history` | catégorielle | Historique de crédit |
| `purpose` | catégorielle | Motif du crédit (voiture, meubles, etc.) |
| `credit_amount` | numérique | Montant du crédit |
| `savings_status` | catégorielle | Épargne |
| `employment` | catégorielle | Ancienneté dans l'emploi actuel |
| `installment_commitment` | numérique | Taux de remboursement en % du revenu disponible |
| `personal_status` | catégorielle | Situation personnelle (sexe et état civil) |
| `other_parties` | catégorielle | Autres débiteurs ou garants |
| `residence_since` | numérique | Ancienneté de résidence (années) |
| `property_magnitude` | catégorielle | Patrimoine le plus important |
| `age` | numérique | Âge (années) |
| `other_payment_plans` | catégorielle | Autres plans de paiement |
| `housing` | catégorielle | Logement (propriétaire, locataire, gratuit) |
| `existing_credits` | numérique | Nombre de crédits existants dans la banque |
| `job` | catégorielle | Type d'emploi |
| `num_dependents` | numérique | Nombre de personnes à charge |
| `own_telephone` | catégorielle | Téléphone enregistré |
| `foreign_worker` | catégorielle | Travailleur étranger |

**Matrice de coûts.** La documentation du jeu de données propose la matrice suivante :

| | Prédit : bon risque (accepter) | Prédit : mauvais risque (refuser) |
|---|---|---|
| **Réel : bon risque** | 0 | **1** (client perdu) |
| **Réel : mauvais risque** | **5** (prêt non remboursé) | 0 |

Accepter un mauvais risque coûte donc **cinq fois plus** que refuser un bon client. Ces coûts sont des hypothèses : nous étudierons plus loin l'effet de les modifier.

Le chargement se fait directement depuis Internet, sans téléchargement manuel. Les données sont d'abord demandées à OpenML (qui fournit des noms de modalités lisibles), avec un repli sur l'archive UCI officielle.

In [ ]:
# Importation des bibliothèques
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, cross_validate, cross_val_predict,
                                     learning_curve, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             confusion_matrix, classification_report)

# Reproductibilité et style des graphiques
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Chargement du jeu de données German Credit (UCI 144)
NOMS = ["checking_status", "duration", "credit_history", "purpose", "credit_amount",
        "savings_status", "employment", "installment_commitment", "personal_status",
        "other_parties", "residence_since", "property_magnitude", "age",
        "other_payment_plans", "housing", "existing_credits", "job", "num_dependents",
        "own_telephone", "foreign_worker", "class"]
URL_UCI = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"

def charger_credit():
    # Source 1 : OpenML (mêmes données, modalités lisibles)
    try:
        d = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
        for col in d.columns:
            if str(d[col].dtype) == "category":
                d[col] = d[col].astype(str)
        return d, "OpenML (credit-g)"
    except Exception as erreur:
        print("OpenML indisponible :", erreur)
    # Source 2 : archive UCI officielle (modalités codées A11, A12, ...)
    with urllib.request.urlopen(URL_UCI) as reponse:
        archive = zipfile.ZipFile(io.BytesIO(reponse.read()))
    d = pd.read_csv(archive.open("german.data"), sep=" ", header=None, names=NOMS)
    d["class"] = d["class"].map({1: "good", 2: "bad"})
    return d, "UCI 144 (archive officielle)"

df, source = charger_credit()
print("Source :", source)
print("Dimensions :", df.shape)
df.head()

## 3. Première exploration

Avant toute modélisation, on examine la structure des données : types des colonnes, valeurs manquantes et statistiques descriptives. Ces étapes servent à repérer les problèmes de qualité qui affecteront le modèle.

Nous créons aussi la **cible numérique** `mauvais_risque` (1 si mauvais risque, 0 sinon). Le mauvais risque est la classe « positive » : c'est l'événement que la banque cherche à détecter.

In [ ]:
# Cible numérique : 1 = mauvais risque (classe à détecter), 0 = bon risque
df["mauvais_risque"] = (df["class"] == "bad").astype(int)
df["risque"] = df["mauvais_risque"].map({0: "Bon risque", 1: "Mauvais risque"})

# Types des colonnes et valeurs manquantes
df.drop(columns=["risque"]).info()
print("\nValeurs manquantes au total :", int(df.isna().sum().sum()))

In [ ]:
# Statistiques descriptives des variables numériques
variables_num = ["duration", "credit_amount", "installment_commitment",
                 "residence_since", "age", "existing_credits", "num_dependents"]
df[variables_num].describe().round(1)

**Lecture.** Le jeu de données est complet : aucune valeur manquante, ce qui simplifie la préparation. Les unités sont très différentes : le montant du crédit se compte en milliers, alors que la durée est en mois (quelques dizaines) et que plusieurs variables ne prennent que quelques valeurs entières (de 1 à 4). Cela justifiera une **mise à l'échelle** avant les modèles sensibles aux distances ou à l'amplitude des variables, comme la régression logistique. On remarque aussi que la moyenne du montant dépasse nettement sa médiane : la distribution est **asymétrique à droite** (quelques très gros prêts).

## 4. Visualisation des données

### 4.1 Répartition des classes

Il faut vérifier si les deux classes sont équilibrées. Un déséquilibre rend l'exactitude trompeuse : un modèle qui répond toujours « bon risque » aurait déjà une bonne note.

In [ ]:
# Diagramme en barres du nombre de dossiers par classe
fig, ax = plt.subplots(figsize=(5.5, 4))
sns.countplot(data=df, x="risque", order=["Bon risque", "Mauvais risque"],
              hue="risque", legend=False, ax=ax)
for patch in ax.patches:
    ax.annotate(f"{int(patch.get_height())}",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.set_title("Nombre de dossiers par classe")
ax.set_xlabel("")
ax.set_ylabel("Effectif")
plt.tight_layout()
plt.show()

**Lecture.** Les classes sont **déséquilibrées** : 700 bons risques contre 300 mauvais (30 %). Conséquence directe : un classifieur qui prédit toujours « bon risque » obtient **70 % d'exactitude** sans rien apprendre. Nous utiliserons donc une **validation stratifiée**, qui conserve ces proportions dans chaque pli, et nous évaluerons les modèles avec l'**AUC** et le **coût**, pas seulement l'exactitude.

### 4.2 Distributions des variables numériques selon le risque

Un histogramme par classe montre si une variable aide à distinguer bons et mauvais payeurs : plus les distributions sont séparées, plus la variable est informative.

In [ ]:
# Histogrammes superposés de quatre variables numériques, par classe
titres_num = {"duration": "Durée du crédit (mois)",
              "credit_amount": "Montant du crédit",
              "age": "Âge (années)",
              "installment_commitment": "Taux de remboursement (% du revenu)"}

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (var, titre) in zip(axes.ravel(), titres_num.items()):
    sns.histplot(data=df, x=var, hue="risque", stat="density", common_norm=False,
                 kde=(var != "installment_commitment"), alpha=0.45, ax=ax,
                 discrete=(var == "installment_commitment"))
    ax.set_title(titre)
    ax.set_xlabel("")
plt.suptitle("Distribution des variables numériques selon le risque", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Lecture.** Les mauvais payeurs ont tendance à emprunter **plus longtemps** et pour des **montants plus élevés**, et ils sont un peu plus jeunes. Mais les distributions se **chevauchent largement** : aucune variable numérique ne suffit à elle seule. Contrairement à un problème où les classes forment des groupes presque disjoints, ce problème est intrinsèquement bruité. On ne doit donc pas s'attendre à une exactitude proche de 100 %.

### 4.3 Taux de mauvais risque selon les variables catégorielles

Pour une variable catégorielle, on calcule la **part de mauvais risques dans chaque modalité**. La ligne pointillée marque le taux global (30 %) : une modalité très au-dessus est un signal d'alerte, une modalité très en dessous est rassurante.

In [ ]:
# Taux de mauvais risque par modalité, pour quatre variables catégorielles
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, var in zip(axes.ravel(), ["checking_status", "credit_history", "purpose", "savings_status"]):
    taux = df.groupby(var)["mauvais_risque"].mean().sort_values()
    ax.barh(taux.index.astype(str), taux.values, color=sns.color_palette("Set2")[1])
    ax.axvline(df["mauvais_risque"].mean(), color="grey", linestyle="--")
    ax.set_xlim(0, 1)
    ax.set_title(var)
    ax.set_xlabel("Part de mauvais risques")
plt.suptitle("Taux de mauvais risque par modalité (pointillé : taux global)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Lecture.** La **situation du compte courant** (`checking_status`) est très discriminante : les clients sans compte ou avec un solde confortable ont un taux de défaut nettement inférieur au taux global, ceux dont le compte est négatif nettement supérieur. L'historique de crédit et l'épargne vont dans le même sens. Les variables comportementales et financières sont donc informatives.

### 4.4 Boîtes à moustaches et corrélations

Les boîtes à moustaches résument la médiane, les quartiles et les valeurs extrêmes. La matrice de corrélation indique les liens linéaires entre variables numériques (cible incluse) : deux variables très corrélées apportent une information redondante.

In [ ]:
# Boîtes à moustaches par classe (échelle logarithmique pour le montant)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, var, titre in zip(axes, ["duration", "credit_amount", "age"],
                          ["Durée (mois)", "Montant (échelle log)", "Âge (années)"]):
    sns.boxplot(data=df, x="risque", y=var, hue="risque", legend=False, ax=ax,
                order=["Bon risque", "Mauvais risque"], hue_order=["Bon risque", "Mauvais risque"])
    ax.set_title(titre)
    ax.set_xlabel("")
    ax.set_ylabel("")
    if var == "credit_amount":
        ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation des variables numériques et de la cible
fig, ax = plt.subplots(figsize=(7.5, 6))
corr = df[variables_num + ["mauvais_risque"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Corrélations entre variables numériques")
plt.tight_layout()
plt.show()

**Lecture.** La durée et le montant du crédit sont **corrélés entre eux** (un long prêt est souvent un gros prêt) : ils partagent une partie de leur information. Leurs corrélations avec la cible restent **modestes** (de l'ordre de 0,1 à 0,2) : ce sont des signaux faibles, qui ne prennent de la valeur qu'en combinaison avec les variables catégorielles. Cette redondance n'est pas gênante pour un arbre ou une forêt, mais justifie de régulariser une régression logistique.

## 5. Préparation des données

Règles à respecter :

- **Ne jamais utiliser le jeu de test pendant la préparation.** Toute transformation (moyenne, écart-type, catégories) est apprise sur l'entraînement uniquement.
- **Valeurs manquantes.** Il n'y en a pas ici : rien à supprimer ni à imputer.
- **Encoder les variables catégorielles.** Un modèle ne lit pas le texte : chaque modalité devient une colonne 0/1 (*one-hot encoding*).
- **Standardiser les variables numériques** : $z = \dfrac{x - \bar x}{s}$, moyenne nulle et écart-type 1.
- **Découpage stratifié** pour que la part de mauvais risques (30 %) soit la même dans l'entraînement et dans le test.

Ces étapes sont regroupées dans un `Pipeline` : la même chaîne est alors appliquée à l'entraînement, à la validation et au test.

In [ ]:
# 1) Séparation entrées / cible
X = df.drop(columns=["class", "mauvais_risque", "risque"])
y = df["mauvais_risque"]
variables_cat = [c for c in X.columns if c not in variables_num]
print(len(variables_num), "variables numériques,", len(variables_cat), "variables catégorielles")

# 2) Découpage stratifié : 75 % entraînement, 25 % test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Entraînement :", X_train.shape[0], "| Test :", X_test.shape[0])
print("Part de mauvais risques -> entraînement :", round(y_train.mean(), 3),
      "| test :", round(y_test.mean(), 3))

# 3) Prétraitement : standardisation des numériques, encodage des catégorielles
preprocesseur = ColumnTransformer([
    ("num", StandardScaler(), variables_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), variables_cat),
])

## 6. Entraînement et comparaison de modèles

### 6.1 Principe de la comparaison

Nous comparons plusieurs algorithmes, du plus simple au plus complexe. La **référence** (*baseline*) est un classifieur qui ignore les données et prédit toujours la classe majoritaire. Tout modèle utile doit faire mieux que cette référence.

Deux métriques sont suivies :

- l'**exactitude** : part de dossiers bien classés ;
- l'**AUC** (aire sous la courbe ROC) : probabilité qu'un mauvais payeur tiré au hasard reçoive un score de risque plus élevé qu'un bon payeur tiré au hasard. Elle vaut 0,5 pour un modèle sans pouvoir de discrimination et 1 pour un modèle parfait. Elle **ne dépend pas du seuil** de décision, ce qui la rend adaptée à un problème déséquilibré.

Pour comparer équitablement, on utilise une **validation croisée à 5 plis sur l'ensemble d'entraînement**. Le jeu de test reste de côté jusqu'à l'évaluation finale.

In [ ]:
# Modèles candidats, chacun précédé du même prétraitement
modeles = {
    "Référence (classe majoritaire)": DummyClassifier(strategy="most_frequent"),
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=300, random_state=42),
}

pipelines = {nom: Pipeline([("prep", preprocesseur), ("modele", m)])
             for nom, m in modeles.items()}

validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats = []
for nom, pipe in pipelines.items():
    cv = cross_validate(pipe, X_train, y_train, cv=validation,
                        scoring={"auc": "roc_auc", "acc": "accuracy"})
    resultats.append({"Modèle": nom,
                      "AUC moyenne": cv["test_auc"].mean(),
                      "Écart-type AUC": cv["test_auc"].std(),
                      "Exactitude moyenne": cv["test_acc"].mean()})

tableau = pd.DataFrame(resultats).sort_values("AUC moyenne", ascending=False)
tableau.round(3)

In [ ]:
# Graphique : AUC moyenne en validation croisée, avec barres d'erreur
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(tableau["Modèle"], tableau["AUC moyenne"],
        xerr=tableau["Écart-type AUC"], capsize=5,
        color=sns.color_palette("Set2", len(tableau)))
ax.set_xlim(0.4, 1.0)
ax.invert_yaxis()
ax.axvline(0.5, color="grey", linestyle="--", label="aucun pouvoir de discrimination (0,5)")
for i, val in enumerate(tableau["AUC moyenne"]):
    ax.text(val + 0.01, i, f"{val:.3f}", va="center")
ax.set_xlabel("AUC (validation croisée à 5 plis)")
ax.set_title("Comparaison des modèles")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Lecture.** Observez d'abord la **référence** : son exactitude est d'environ 0,70 (elle prédit toujours « bon risque »), mais son AUC vaut exactement 0,5. C'est la démonstration que l'exactitude seule est trompeuse ici : un modèle qui n'a rien appris obtient déjà 70 %. Les modèles appris améliorent surtout l'**AUC**. Attendez-vous à des AUC de l'ordre de 0,75 à 0,80 pour la régression logistique et la forêt aléatoire, un peu moins pour l'arbre de profondeur 4. Ce niveau est correct sans être exceptionnel : le problème est bruité. Notez vos valeurs, elles serviront de repère pour la suite. Les écarts-types indiquent la stabilité : sur seulement 750 dossiers d'entraînement, de petits écarts d'AUC entre deux modèles ne sont pas forcément significatifs.

## 7. Évaluation finale sur le jeu de test

Le modèle choisi est entraîné sur tout l'ensemble d'entraînement, puis évalué **une seule fois** sur le jeu de test. Nous sélectionnons automatiquement le modèle ayant la meilleure AUC en validation croisée.

### 7.1 Matrice de confusion au seuil habituel de 0,5

Un classifieur produit une **probabilité de mauvais risque**. Pour décider, il faut la comparer à un **seuil** : on refuse le dossier si la probabilité atteint ce seuil. Le seuil par défaut est 0,5. Voyons ce qu'il donne.

In [ ]:
# Sélection du meilleur modèle selon l'AUC en validation croisée
meilleur_nom = tableau.iloc[0]["Modèle"]
meilleur_pipe = pipelines[meilleur_nom]
meilleur_pipe.fit(X_train, y_train)

# Probabilité de mauvais risque sur le test (colonne 1 = classe "mauvais risque")
p_test = meilleur_pipe.predict_proba(X_test)[:, 1]
y_pred_05 = (p_test >= 0.5).astype(int)

print("Meilleur modèle :", meilleur_nom)
print(f"AUC sur le jeu de test : {roc_auc_score(y_test, p_test):.3f}")
print(f"Exactitude au seuil 0,5 : {accuracy_score(y_test, y_pred_05):.3f}\n")
print(classification_report(y_test, y_pred_05, target_names=["Bon risque", "Mauvais risque"]))

In [ ]:
# Matrice de confusion : valeurs absolues et proportions par ligne
noms_classes = ["Bon risque", "Mauvais risque"]
cm = confusion_matrix(y_test, y_pred_05, labels=[0, 1])
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=noms_classes,
            yticklabels=noms_classes, cbar=False, ax=axes[0])
axes[0].set_title("Effectifs (seuil 0,5)")
axes[0].set_xlabel("Décision du modèle")
axes[0].set_ylabel("Situation réelle")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", xticklabels=noms_classes,
            yticklabels=noms_classes, vmin=0, vmax=1, cbar=False, ax=axes[1])
axes[1].set_title("Proportions par situation réelle (rappel)")
axes[1].set_xlabel("Décision du modèle")
axes[1].set_ylabel("Situation réelle")
plt.tight_layout()
plt.show()

**Lecture.** La case en bas à gauche (mauvais risque classé « bon risque ») contient les **mauvais payeurs acceptés par erreur** : ce sont les erreurs les plus coûteuses (coût 5). La case en haut à droite contient les **bons clients refusés** (coût 1). Au seuil de 0,5, le modèle laisse typiquement passer une **part importante** des mauvais payeurs : le **rappel** de la classe « mauvais risque » est nettement plus faible que celui de la classe « bon risque ». Le seuil 0,5 est pensé pour des erreurs de même coût, ce qui n'est pas notre cas. Il faut donc **déplacer le seuil**.

### 7.2 Choisir le seuil de décision à partir des coûts

Pour un seuil $t$, on calcule le **coût moyen par dossier** :

$$\text{coût}(t) = \frac{5 \times \text{FN}(t) + 1 \times \text{FP}(t)}{n}$$

où FN est le nombre de mauvais payeurs acceptés et FP le nombre de bons clients refusés. On choisit le seuil qui minimise ce coût.

**Point méthodologique important.** Le seuil est choisi **sans regarder le jeu de test** : on utilise les probabilités prédites *hors pli* sur l'ensemble d'entraînement (`cross_val_predict`), c'est-à-dire des prédictions faites par un modèle qui n'a pas vu le dossier concerné. Le jeu de test sert ensuite uniquement à vérifier le résultat. Choisir le seuil sur le test reviendrait à tricher.

Si les probabilités sont bien calibrées, la théorie donne un seuil optimal de $\dfrac{c_{FP}}{c_{FP}+c_{FN}} = \dfrac{1}{6} \approx 0{,}17$ : on refuse dès que la probabilité de défaut dépasse environ 17 %.

In [ ]:
# Matrice de coûts (hypothèse de la documentation du jeu de données)
C_FN = 5   # accepter un mauvais risque
C_FP = 1   # refuser un bon client

def cout_moyen(y_vrai, proba, seuil, c_fn=C_FN, c_fp=C_FP):
    # Coût moyen par dossier si on refuse tout dossier dont proba >= seuil
    y_vrai = np.asarray(y_vrai)
    decision = (np.asarray(proba) >= seuil).astype(int)
    fn = np.sum((y_vrai == 1) & (decision == 0))
    fp = np.sum((y_vrai == 0) & (decision == 1))
    return (c_fn * fn + c_fp * fp) / len(y_vrai)

# Probabilités hors pli sur l'entraînement (aucune donnée de test utilisée)
p_oof = cross_val_predict(meilleur_pipe, X_train, y_train, cv=validation,
                          method="predict_proba")[:, 1]

seuils = np.linspace(0.02, 0.98, 97)
couts_oof = np.array([cout_moyen(y_train, p_oof, s) for s in seuils])
couts_test = np.array([cout_moyen(y_test, p_test, s) for s in seuils])
seuil_opt = seuils[couts_oof.argmin()]
print(f"Seuil optimal (choisi sur l'entraînement) : {seuil_opt:.2f}")
print(f"Seuil théorique 1/(1+{C_FN}) : {C_FP / (C_FP + C_FN):.2f}")

In [ ]:
# Courbe de coût selon le seuil de décision
cout_tout_accepter = C_FN * y_train.mean()          # on accepte tout le monde
cout_tout_refuser = C_FP * (1 - y_train.mean())     # on refuse tout le monde

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(seuils, couts_oof, label="Entraînement (prédictions hors pli)", color="tab:blue")
ax.plot(seuils, couts_test, label="Test", color="tab:orange", linestyle="--")
ax.axvline(seuil_opt, color="green", linestyle=":", label=f"seuil optimal = {seuil_opt:.2f}")
ax.axvline(0.5, color="grey", linestyle=":", label="seuil par défaut = 0,5")
ax.axhline(cout_tout_accepter, color="red", alpha=0.5, label="tout accepter")
ax.axhline(cout_tout_refuser, color="purple", alpha=0.5, label="tout refuser")
ax.set_xlabel("Seuil de décision (on refuse si probabilité de défaut ≥ seuil)")
ax.set_ylabel("Coût moyen par dossier")
ax.set_title(f"Courbe de coût selon le seuil : {meilleur_nom}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des stratégies sur le jeu de test (coût moyen par dossier)
comparaison = pd.DataFrame({
    "Stratégie": ["Tout accepter", "Tout refuser", "Modèle, seuil 0,5",
                  f"Modèle, seuil optimal ({seuil_opt:.2f})"],
    "Coût moyen par dossier (test)": [
        cout_moyen(y_test, np.zeros(len(y_test)), 1.01),   # personne refusé
        cout_moyen(y_test, np.ones(len(y_test)), 0.5),     # tout le monde refusé
        cout_moyen(y_test, p_test, 0.5),
        cout_moyen(y_test, p_test, seuil_opt)],
})
comparaison.round(3)

In [ ]:
# Matrice de confusion au seuil optimal
y_pred_opt = (p_test >= seuil_opt).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_opt, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5.5, 4.4))
sns.heatmap(cm_opt, annot=True, fmt="d", cmap="Blues", xticklabels=noms_classes,
            yticklabels=noms_classes, cbar=False, ax=ax)
ax.set_title(f"Matrice de confusion (seuil optimal {seuil_opt:.2f})")
ax.set_xlabel("Décision du modèle")
ax.set_ylabel("Situation réelle")
plt.tight_layout()
plt.show()
print(f"Rappel des mauvais risques : {cm_opt[1, 1] / cm_opt[1].sum():.2f}"
      f" (au seuil 0,5 : {cm[1, 1] / cm[1].sum():.2f})")
print(f"Part de bons clients refusés : {cm_opt[0, 1] / cm_opt[0].sum():.2f}"
      f" (au seuil 0,5 : {cm[0, 1] / cm[0].sum():.2f})")

**Lecture.** La courbe de coût a une forme en **cuvette** : un seuil trop élevé laisse passer trop de mauvais payeurs (coût élevé à droite), un seuil trop bas refuse presque tout le monde (coût élevé à gauche, vers la stratégie « tout refuser »). Le seuil optimal se situe **bien en dessous de 0,5**, dans la zone de 0,15 à 0,35 environ, ce qui est cohérent avec la valeur théorique de 0,17. En l'adoptant, le modèle **détecte beaucoup plus de mauvais payeurs** au prix de **plus de bons clients refusés**, et le **coût moyen baisse** par rapport au seuil 0,5. Comparez aussi avec les deux stratégies sans modèle : le modèle doit les battre pour justifier son existence.

**Prudence.** Le jeu de test ne contient que 250 dossiers dont environ 75 mauvais payeurs : la courbe de test est **bruitée** et son minimum ne coïncide pas exactement avec celui de l'entraînement. C'est normal. Un écart de coût de quelques centièmes entre deux seuils voisins n'est pas significatif.

### 7.3 Analyse de sensibilité à la matrice de coûts

Le rapport de coûts de 5 pour 1 est une **hypothèse**, pas une vérité. Une banque réelle devrait estimer ce rapport à partir de ses pertes effectives et de ses marges. Le seuil optimal en dépend directement : si le rapport change, la décision change. Il est donc indispensable de présenter une **analyse de sensibilité**.

In [ ]:
# Seuil optimal et coût sur le test pour différents rapports de coûts
rapports = [1, 2, 3, 5, 10, 20]
lignes = []
for r in rapports:
    couts_r = [cout_moyen(y_train, p_oof, s, c_fn=r, c_fp=1) for s in seuils]
    s_opt = seuils[int(np.argmin(couts_r))]
    lignes.append({"Coût FN / coût FP": r,
                   "Seuil optimal (entraînement)": s_opt,
                   "Seuil théorique 1/(1+r)": 1 / (1 + r),
                   "Coût moyen test": cout_moyen(y_test, p_test, s_opt, c_fn=r, c_fp=1)})
sensibilite = pd.DataFrame(lignes)
display(sensibilite.round(3))

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(sensibilite["Coût FN / coût FP"], sensibilite["Seuil optimal (entraînement)"],
        "o-", label="Seuil optimal observé")
ax.plot(sensibilite["Coût FN / coût FP"], sensibilite["Seuil théorique 1/(1+r)"],
        "s--", label="Seuil théorique 1/(1+r)")
ax.set_xlabel("Rapport coût FN / coût FP")
ax.set_ylabel("Seuil de décision")
ax.set_title("Le seuil optimal dépend des hypothèses de coût")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Plus accepter un mauvais payeur est coûteux par rapport à refuser un bon client, plus le seuil optimal **baisse** : la banque devient plus prudente. Pour un rapport de 1 (erreurs de même coût), on retrouve un seuil proche de 0,5. Le seuil observé suit globalement la courbe théorique, mais s'en écarte parce que les probabilités du modèle ne sont **pas parfaitement calibrées** (une forêt aléatoire, par exemple, produit des probabilités peu fiables dans les zones extrêmes). Conclusion à retenir : **ne jamais présenter un seuil optimal sans préciser l'hypothèse de coût qui le justifie**.

### 7.4 Courbes ROC

La courbe ROC représente, pour tous les seuils possibles, le taux de mauvais payeurs détectés (en ordonnée) en fonction du taux de bons clients refusés à tort (en abscisse). Plus la courbe s'approche du coin supérieur gauche, meilleur est le modèle. La diagonale correspond au hasard.

In [ ]:
# Courbes ROC de tous les modèles sur le jeu de test
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for nom, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, label=f"{nom} (AUC = {roc_auc_score(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("Taux de bons clients refusés à tort")
ax.set_ylabel("Taux de mauvais payeurs détectés")
ax.set_title("Courbes ROC sur le jeu de test")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

**Lecture.** La référence suit la diagonale. Les trois autres modèles s'en détachent, mais leurs courbes sont **proches les unes des autres** : sur un jeu de test de 250 dossiers, l'ordre exact entre régression logistique et forêt aléatoire peut changer d'un découpage à l'autre. Choisir « le meilleur » modèle sur de si petits écarts demande de la prudence. Chaque point de la courbe correspond à un seuil : le seuil optimal de la section précédente est un point précis de cette courbe, situé du côté où l'on détecte beaucoup de mauvais payeurs.

### 7.5 Courbe d'apprentissage

La courbe d'apprentissage répond à une question pratique : **le modèle manque-t-il de données, ou de flexibilité ?**

- Si les courbes d'entraînement et de validation convergent vers un score faible : **sous-apprentissage** (biais élevé).
- Si l'écart reste grand : **surapprentissage** (variance élevée) ; plus de données ou une régularisation aideraient.

In [ ]:
# Courbe d'apprentissage du meilleur modèle (AUC)
tailles, scores_train, scores_val = learning_curve(
    meilleur_pipe, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=validation, scoring="roc_auc", n_jobs=-1
)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
for scores, label, couleur in [(scores_train, "Entraînement", "tab:blue"),
                               (scores_val, "Validation", "tab:orange")]:
    moyenne, ecart = scores.mean(axis=1), scores.std(axis=1)
    ax.plot(tailles, moyenne, "o-", color=couleur, label=label)
    ax.fill_between(tailles, moyenne - ecart, moyenne + ecart, alpha=0.2, color=couleur)
ax.set_xlabel("Nombre d'observations d'entraînement")
ax.set_ylabel("AUC")
ax.set_title(f"Courbe d'apprentissage : {meilleur_nom}")
ax.set_ylim(0.5, 1.02)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Lecture.** Comparez l'écart entre les deux courbes. Pour une forêt aléatoire, l'AUC d'entraînement est proche de 1 (le modèle mémorise ses exemples) alors que l'AUC de validation plafonne autour de 0,75 à 0,80 : c'est un signe de **surapprentissage**, ou plutôt de la limite d'information contenue dans les variables. Si la courbe de validation monte encore à droite du graphique, plus de données aideraient. Avec seulement 1 000 dossiers, il serait imprudent de promettre de meilleures performances sans en collecter davantage.

## 8. Interprétation du modèle

Dans le crédit, **comprendre pourquoi un dossier est refusé** n'est pas un luxe : c'est souvent une obligation réglementaire et un besoin métier. Nous regardons deux angles complémentaires.

### 8.1 Importance des variables (forêt aléatoire)

La forêt aléatoire fournit une mesure d'importance : la réduction moyenne de l'impureté de Gini apportée par chaque variable.

In [ ]:
# Importance des variables pour la forêt aléatoire
foret = Pipeline([("prep", preprocesseur),
                  ("modele", RandomForestClassifier(n_estimators=300, random_state=42))])
foret.fit(X_train, y_train)

noms = [n.replace("num__", "").replace("cat__", "")
        for n in foret.named_steps["prep"].get_feature_names_out()]
importances = pd.Series(foret.named_steps["modele"].feature_importances_, index=noms)
top = importances.sort_values().tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
top.plot.barh(ax=ax, color="teal")
for i, val in enumerate(top):
    ax.text(val + 0.001, i, f"{val:.3f}", va="center")
ax.set_xlabel("Importance (réduction moyenne de l'impureté de Gini)")
ax.set_title("15 variables les plus importantes, forêt aléatoire")
plt.tight_layout()
plt.show()

**Lecture.** Les variables numériques continues (montant, âge, durée) apparaissent souvent en tête parce qu'elles offrent **beaucoup de points de coupure possibles** à un arbre : l'importance de Gini a un biais connu en leur faveur. Les modalités du compte courant ressortent aussi. Cette importance dépend du modèle, peut être trompeuse quand des variables sont corrélées, et ne dit **pas dans quel sens** une variable agit. Elle sert à orienter l'analyse, pas à établir une causalité.

### 8.2 Coefficients de la régression logistique

Pour connaître le **sens** de l'effet, on regarde les coefficients de la régression logistique. Un coefficient positif augmente la probabilité de mauvais risque, un coefficient négatif la diminue. Les variables numériques étant standardisées, leurs coefficients sont comparables entre eux.

In [ ]:
# Coefficients de la régression logistique (effet sur le risque de défaut)
logit = Pipeline([("prep", preprocesseur), ("modele", LogisticRegression(max_iter=1000))])
logit.fit(X_train, y_train)

noms_logit = [n.replace("num__", "").replace("cat__", "")
              for n in logit.named_steps["prep"].get_feature_names_out()]
coefs = pd.Series(logit.named_steps["modele"].coef_[0], index=noms_logit)
extremes = pd.concat([coefs.sort_values().head(8), coefs.sort_values().tail(8)])

fig, ax = plt.subplots(figsize=(9, 6))
couleurs = ["tab:green" if v < 0 else "tab:red" for v in extremes]
ax.barh(extremes.index, extremes.values, color=couleurs)
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel("Coefficient (rouge : augmente le risque, vert : le diminue)")
ax.set_title("Régression logistique : 16 effets les plus marqués")
plt.tight_layout()
plt.show()

**Lecture.** Les barres rouges désignent les caractéristiques associées à un **risque accru** (par exemple un compte courant négatif, une durée longue), les barres vertes celles associées à un **risque réduit** (par exemple l'absence de compte courant ou un solde élevé, selon les modalités). Avec la source OpenML, les noms des modalités sont lisibles ; avec le repli UCI, ils sont codés (A11, A12, etc.) et il faut les relire dans la documentation du jeu de données. Attention : comme les modalités d'une même variable sont toutes présentes dans le modèle, les coefficients individuels sont à interpréter **les uns par rapport aux autres**, pas isolément.

> **Point d'éthique.** Le jeu contient des variables sensibles (`personal_status`, qui mêle sexe et état civil, `age`, `foreign_worker`). Utiliser ces variables pour accepter ou refuser un crédit peut être discriminatoire, voire illégal selon les pays. Un modèle performant n'est pas forcément un modèle acceptable. Voir la question 5.

## 9. Prédire un nouveau dossier

Une fois le modèle validé, il sert à évaluer un nouveau dossier. Le pipeline applique automatiquement la même standardisation et le même encodage qu'à l'entraînement. Pour construire un dossier fictif sans se soucier du nom exact des modalités, nous partons d'un dossier du jeu de test et modifions ses variables numériques.

In [ ]:
# Dossier fictif : on part d'un dossier existant et on modifie quelques valeurs
nouveau = X_test.iloc[[0]].copy()
nouveau["duration"] = 36
nouveau["credit_amount"] = 9000
nouveau["age"] = 24
display(nouveau.T)

proba_defaut = meilleur_pipe.predict_proba(nouveau)[0, 1]
decision = "REFUSER" if proba_defaut >= seuil_opt else "ACCEPTER"
print(f"Probabilité estimée de mauvais risque : {proba_defaut:.2f}")
print(f"Seuil de décision (optimal pour un rapport de coûts de {C_FN}) : {seuil_opt:.2f}")
print("Décision :", decision)

## 10. Synthèse et questions

### Ce qu'il faut retenir

1. **Type d'apprentissage** : supervisé, classification binaire à coûts asymétriques. La cible (bon ou mauvais risque) est connue pour chaque dossier.
2. **Démarche** : explorer, prétraiter dans un pipeline, comparer à une référence, valider par validation croisée, évaluer une seule fois sur le test.
3. **L'exactitude trompe** quand les classes sont déséquilibrées : la référence atteint 70 % sans rien apprendre. Utilisez l'AUC pour comparer les modèles et le coût pour décider.
4. **Le seuil de décision est un choix économique**, pas technique. Il se choisit sur des données d'entraînement (prédictions hors pli), jamais sur le test, et dépend d'une hypothèse de coût qu'il faut **justifier et tester** par une analyse de sensibilité.
5. **Limites** : 1 000 dossiers, une seule banque, des données anciennes (années 1990), une matrice de coûts supposée. Ces résultats illustrent une méthode, ils ne se généralisent pas tels quels à un portefeuille réel.

### Questions

1. Pourquoi une exactitude de 70 % ne prouve-t-elle rien dans ce problème ? Quelle exactitude obtiendrait un modèle qui refuse systématiquement tous les dossiers ? Quel serait son coût moyen ?
2. Faites varier `max_depth` de l'arbre de décision (1, 2, 4, 8, 16). Tracez l'AUC en validation et en entraînement. Observez-vous du sous-apprentissage ou du surapprentissage ?
3. Ajoutez `class_weight="balanced"` à la régression logistique. L'AUC change-t-elle ? Le seuil optimal change-t-il ? Pourquoi ces deux réponses sont-elles différentes ?
4. Appliquez une transformation logarithmique à `credit_amount` avant la standardisation. Le résultat s'améliore-t-il pour la régression logistique ? Pour la forêt aléatoire ? Pourquoi ?
5. Retirez les variables `personal_status`, `age` et `foreign_worker` et réentraînez les modèles. Quelle est la perte de performance ? Un modèle qui n'utilise pas explicitement ces variables est-il pour autant équitable ? (Indice : d'autres variables peuvent leur servir de substitut.)
6. Si le coût réel d'un défaut était **20 fois** celui d'un refus injustifié, quel seuil recommanderiez-vous ? Quelle part des bons clients la banque refuserait-elle ? Comment présenteriez-vous ce compromis à un directeur du crédit ?
7. Ce problème pourrait-il être traité en apprentissage non supervisé ? Décrivez une expérience (par exemple un k-means sur les variables standardisées) et ce que vous compareriez à l'étiquette réelle (indice ARI).

> **Pour aller plus loin.** Remplacez la cible par `credit_amount` (valeur réelle) : vous passeriez à un problème de **régression**. Quelles métriques (RMSE, MAE, R²) utiliseriez-vous ? Vous pouvez aussi essayer un modèle de gradient boosting (`HistGradientBoostingClassifier`) et comparer son coût à celui de la régression logistique.